In [1]:
import ants
import SimpleITK as sitk
import nibabel as nib
import numpy as np
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

import os
import time

os.environ['NEURITE_BACKEND'] = 'pytorch'
os.environ['VXM_BACKEND'] = 'pytorch'
import voxelmorph as vxm
import torch

import shutil

In [2]:
""" Volume Upsampling
    Upsample 75um PA to 25um
    Remember to Downsample back to 75um for inference and optimization
"""
# 1. PA Volume Rigid Transformation
 # for old vxm model, it is aligned to pa brain number 241
ifexist = 0

fixed_image = ants.image_read('D:/LabData/RegData/DWCPI_Registration/data/atlas_75_raw.nii') # for new vxm model, it is aligned to 75 um allen brain
#fixed_image = ants.image_read('D:/LabData/RegData/mice_dataset2/vxm_evaluation_new/358/atlas_75_raw.nii')
moving_image = ants.image_read('mice_dataset2/vxm_evaluation_new/385/pa_brain_raw_385.nii')
moving_mask = ants.image_read('mice_dataset2/vxm_evaluation_new/385/pa_brain_mask_385.nii')

if ifexist:
    mytx = {}
    mytx['fwdtransforms'] = []
    mytx['fwdtransforms'].append("D:/LabData/RegData/mice_dataset2/vxm_evaluation_new/244/244_rigid_affine.mat")

    registered_pa = ants.apply_transforms(fixed = fixed_image,
                                              moving = moving_image,
                                              transformlist = mytx['fwdtransforms'],
                                              interpolator = "linear")
    ants.image_write(registered_pa, 'mice_dataset2/vxm_evaluation_new/244/aligned_pa_244.nii')

    registered_pa_mask = ants.apply_transforms(fixed = fixed_image,
                                                   moving = moving_mask,
                                                   transformlist = mytx['fwdtransforms'],
                                                   interpolator="nearestNeighbor")
    ants.image_write(registered_pa_mask, 'mice_dataset2/vxm_evaluation_new/244/aligned_pa_mask_244.nii') 
else:
    similarity_registration_result = ants.registration(
        fixed = fixed_image,
        moving = moving_image,
        type_of_transform = 'Rigid')
    
    rigid_transform = similarity_registration_result['fwdtransforms'][0]

    # Apply the transformation to the moving image
    registered_moving_image = ants.apply_transforms(
        fixed=fixed_image,
        moving=moving_image,
        transformlist=[rigid_transform])

    ants.image_write(registered_moving_image, 'mice_dataset2/vxm_evaluation_new/385/aligned_pa_385.nii')

    # Apply the same affine transformation to the mask
    registered_moving_mask = ants.apply_transforms(
        fixed=fixed_image,
        moving=moving_mask,
        transformlist=[rigid_transform],
        interpolator='nearestNeighbor'  # Use nearest neighbor interpolation for binary masks
        )

    # Save the registered moving mask
    ants.image_write(registered_moving_mask, "mice_dataset2/vxm_evaluation_new/385/aligned_pa_mask_385.nii")

    shutil.copyfile(similarity_registration_result['fwdtransforms'][0], 'mice_dataset2/vxm_evaluation_new/385/385_rigid_affine.mat')


pa_original = nib.load('mice_dataset2/vxm_evaluation_new/385/aligned_pa_385.nii')
pa_image = pa_original.get_fdata()

# upsample factor
upsample_factor = 3

# upsample the pa volume
pa_upsampled = zoom(pa_image, upsample_factor, order=1)

# create the new pa file
upsampled_nii = nib.Nifti1Image(pa_upsampled, pa_original.affine)

# save the upsampled nii file
nib.save(upsampled_nii, 'mice_dataset2/vxm_evaluation_new/385/aligned_upsampled_raw_385.nii')

In [3]:
# 2. Atlas and Annotation Affine Transformation (Result in 25um)
fixed_image = ants.image_read('mice_dataset2/vxm_evaluation_new/385/aligned_upsampled_raw_385.nii')

moving_image = ants.image_read('mice_dataset2/vxm_evaluation_new/P56_padded_no_cut.nii')
moving_mask = ants.image_read('mice_dataset2/vxm_evaluation_new/P56_padded_mask_no_cut.nii')
#moving_image = ants.image_read('mice_dataset2/vxm_evaluation_new/358/atlas_25_raw.nii')
#moving_mask = ants.image_read('mice_dataset2/vxm_evaluation_new/358/atlas_25_mask.nii')  

# Perform affine registration
registration_result = ants.registration(
    fixed=fixed_image,
    moving=moving_image,
    type_of_transform='Affine'
)

# Save the transformation matrix
affine_transform = registration_result['fwdtransforms'][0]

# Apply the transformation to the moving image
registered_moving_image = ants.apply_transforms(
    fixed=fixed_image,
    moving=moving_image,
    transformlist=[affine_transform]
)

# Save the registered moving image (Pass to downstream)
ants.image_write(registered_moving_image, "mice_dataset2/vxm_evaluation_new/385/not_cropped_upsampled_atlas_385.nii")

# Apply the same affine transformation to the mask
registered_moving_mask = ants.apply_transforms(
    fixed=fixed_image,
    moving=moving_mask,
    transformlist=[affine_transform],
    interpolator='nearestNeighbor'  # Use nearest neighbor interpolation for binary masks
)

# Save the registered moving mask
ants.image_write(registered_moving_mask, "mice_dataset2/vxm_evaluation_new/385/not_cropped_upsampled_atlas_annotation_385.nii")

print("Registration and mask transformation completed successfully.")

Registration and mask transformation completed successfully.


In [6]:
""" Three Volumes Should Be Cropped (25um):
aligned_upsampled_raw_*.nii --> aligned_pa_25_cut_*.nii
not_cropped_upsampled_atlas_*.nii --> atlas_25_*.nii
not_cropped_upsampled_aligned_annotation_*.nii --> annotation_25_*.nii
"""
# Load the NIfTI file
#nii_file = 'mice_dataset2/vxm_evaluation_new/385/aligned_upsampled_raw_385.nii'
#nii_file = 'mice_dataset2/vxm_evaluation_new/385/not_cropped_upsampled_atlas_385.nii'
nii_file = 'mice_dataset2/vxm_evaluation_new/385/not_cropped_upsampled_atlas_annotation_385.nii'

img = nib.load(nii_file)

# Get the image data and affine transformation matrix
data = img.get_fdata()
affine = img.affine

# Define the cropping region
x_start, x_end = 147, 627
y_start, y_end = 60, 684
z_start, z_end = 222, 558

# Crop the image data
cropped_data = data[x_start:x_end, y_start:y_end, z_start:z_end]

# Update the affine transformation matrix
new_affine = np.copy(affine)
new_affine[:3, 3] += affine[:3, :3] @ np.array([x_start, y_start, z_start])

# Save the cropped image to a new NIfTI file
cropped_img = nib.Nifti1Image(cropped_data, new_affine)
nib.save(cropped_img, 'mice_dataset2/vxm_evaluation_new/385/annotation_25_385.nii')
#nib.save(cropped_img, 'mice_dataset2/vxm_evaluation_new/385/atlas_25_385.nii')
#nib.save(cropped_img, 'mice_dataset2/vxm_evaluation_new/385/aligned_pa_25_cut_385.nii')

In [7]:
# downsample factor
downsample_factor = 1 / 3

# load upsampled pa volume and atlas
pa_25 = nib.load('mice_dataset2/vxm_evaluation_new/385/aligned_upsampled_raw_385.nii')
atlas_25 = nib.load('mice_dataset2/vxm_evaluation_new/385/not_cropped_upsampled_atlas_385.nii')
annotation_25 = nib.load('mice_dataset2/vxm_evaluation_new/385/not_cropped_upsampled_atlas_annotation_385.nii')

pa_data = pa_25.get_fdata()
atlas_data = atlas_25.get_fdata()
annotation_data = annotation_25.get_fdata()

# downsample the volumes
pa_75 = zoom(pa_data, downsample_factor, order = 1)
atlas_75 = zoom(atlas_data, downsample_factor, order = 1)
annotation_75 = zoom(annotation_data, downsample_factor, order = 0)

# create the downsample file 
downsampled_pa = nib.Nifti1Image(pa_75, pa_25.affine)
downsampled_atlas = nib.Nifti1Image(atlas_75, atlas_25.affine)
downsampled_annotation = nib.Nifti1Image(annotation_75, annotation_25.affine)

# save nii file
nib.save(downsampled_pa, 'mice_dataset2/vxm_evaluation_new/385/not_cropped_aligned_pa_brain_raw_385.nii')
nib.save(downsampled_atlas, 'mice_dataset2/vxm_evaluation_new/385/not_cropped_aligned_atlas_385.nii')
nib.save(downsampled_annotation, 'mice_dataset2/vxm_evaluation_new/385/not_cropped_aligned_annotation_385.nii')

In [11]:
""" Three Volumes Should Be Cropped(75um):
not_cropped_aligned_pa_brain_raw_*.nii --> aligned_pa_cut_*.nii
not_cropped_aligned_atlas_*.nii --> atlas_*.nii
not_cropped_aligned_annotation_*.nii --> annotation_*.nii
aligned_pa_mask_*.nii --> aligned_pa_mask_cut_*.nii
"""
# Load the NIfTI file
#nii_file = 'mice_dataset2/vxm_evaluation_new/385/not_cropped_aligned_pa_brain_raw_385.nii'
#nii_file = 'mice_dataset2/vxm_evaluation_new/385/not_cropped_aligned_atlas_385.nii'
#nii_file = 'mice_dataset2/vxm_evaluation_new/385/not_cropped_aligned_annotation_385.nii'
nii_file = 'mice_dataset2/vxm_evaluation_new/385/aligned_pa_mask_385.nii'

img = nib.load(nii_file)

# Get the image data and affine transformation matrix
data = img.get_fdata()
affine = img.affine

# Define the cropping region
x_start, x_end = 49, 209
y_start, y_end = 20, 228
z_start, z_end = 74, 186

# Crop the image data
cropped_data = data[x_start:x_end, y_start:y_end, z_start:z_end]

# Update the affine transformation matrix
new_affine = np.copy(affine)
new_affine[:3, 3] += affine[:3, :3] @ np.array([x_start, y_start, z_start])

# Save the cropped image to a new NIfTI file
cropped_img = nib.Nifti1Image(cropped_data, new_affine)
#nib.save(cropped_img, 'mice_dataset2/vxm_evaluation_new/385/annotation_385.nii')
#nib.save(cropped_img, 'mice_dataset2/vxm_evaluation_new/385/atlas_385.nii')
#nib.save(cropped_img, 'mice_dataset2/vxm_evaluation_new/385/aligned_pa_cut_385.nii')
nib.save(cropped_img, 'mice_dataset2/vxm_evaluation_new/385/aligned_pa_mask_cut_385.nii')

In [12]:
""" Normalization
1. PA volume should be normalized to [0,1] aligned_pa_cut_*.nii --> normalized_pa_*.nii
2. Atlas volume should be normalized to [0,1] atlas_*.nii --> normalized_atlas_*.nii

"""
def normalize_image_with_mask(image, mask):
    img_array = image
    mask_array = mask
    
    # Apply mask to get foreground values
    foreground_values = img_array[mask_array > 0]
    
    # Normalize only the foreground values
    min_v = np.min(foreground_values)
    max_v = np.max(foreground_values)
    normalized_foreground = (foreground_values - min_v) / (max_v-min_v)
    
    # Replace the foreground values in the image
    normalized_image_array = np.copy(img_array)
    normalized_image_array[mask_array > 0] = normalized_foreground
    
    
    return normalized_image_array

image_pa = sitk.ReadImage('mice_dataset2/vxm_evaluation_new/385/aligned_pa_cut_385.nii')
pa_image = sitk.GetArrayFromImage(image_pa)
pa_mask_file = sitk.ReadImage('mice_dataset2/vxm_evaluation_new/385/aligned_pa_mask_cut_385.nii')
pa_mask = sitk.GetArrayFromImage(pa_mask_file)
pa_mask[pa_mask != 0] = 1

image_allen = sitk.ReadImage('mice_dataset2/vxm_evaluation_new/385/atlas_385.nii')
allen_annotation = sitk.ReadImage('mice_dataset2/vxm_evaluation_new/385/annotation_385.nii')
allen_image =sitk.GetArrayFromImage(image_allen)
allen_annotation = sitk.GetArrayFromImage(allen_annotation)
allen_mask = allen_annotation
allen_mask[allen_mask != 0] = 1
allen_image = np.multiply(allen_image, allen_mask)

normalized_pa = normalize_image_with_mask(pa_image, pa_mask)
normalized_pa[pa_mask ==0] = 0
normalized_allen = normalize_image_with_mask(allen_image, allen_mask)
normalized_allen[allen_mask == 0] = 0

normalized_pa_img = sitk.GetImageFromArray(normalized_pa)
normalized_pa_img.CopyInformation(image_pa)
normalized_allen_img = sitk.GetImageFromArray(normalized_allen)
normalized_allen_img.CopyInformation(image_allen)

sitk.WriteImage(normalized_pa_img, 'mice_dataset2/vxm_evaluation_new/385/normalized_pa_385.nii')
sitk.WriteImage(normalized_allen_img, 'mice_dataset2/vxm_evaluation_new/385/normalized_atlas_385.nii')

In [ ]:
""" Instance-specific Optimization
generating optimized_atlas_*.pt for inference 
"""
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# prepare model folder
model_dir = 'mice_dataset2/test_4_optimization'
os.makedirs(model_dir, exist_ok=True)

# device handling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# enabling cudnn determinism appears to speed up training by a lot
torch.backends.cudnn.deterministic = True

# training parameters
int_steps = 7
int_downsize = 2
lr = 1e-4
weight = 1
initial_epoch = 0
epochs = 300
steps_per_epoch = 1

moving = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/385/normalized_atlas_385.nii', add_batch_axis=True, add_feat_axis=True)
fixed, fixed_affine = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/385/normalized_pa_385.nii', add_batch_axis=True, add_feat_axis=True, ret_affine=True)
input_moving = torch.from_numpy(moving).float().permute(0, 4, 1, 2, 3)
input_fixed = torch.from_numpy(fixed).float().permute(0, 4, 1, 2, 3)
inputs = [input_moving, input_fixed]
outputs = [input_fixed]
outputs.append(torch.from_numpy(np.zeros((1, 160, 208, 112, 1))).float().permute(0, 4, 1, 2, 3))

model_inputs = [inputs, outputs]
model_input, model_y_true = model_inputs

# prepare the model for training and send to device
model = vxm.networks.VxmDense.load('mice_dataset2/vxm_evaluation_new/1e4_batch4/1000.pt', device)
model.to(device)
# set optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# prepare image loss
image_loss_func = vxm.losses.NCC().loss
losses = [image_loss_func]
weights = [1]

# prepare deformation loss
losses += [vxm.losses.Grad('l2', loss_mult=int_downsize).loss]
weights += [weight]

#train
epoch_total_loss_list = []
epoch_similarity_loss_list = []
epoch_deformation_loss_list = []
# training loops

for epoch in range(initial_epoch, epochs):

    model.train()

    epoch_loss = []
    epoch_total_loss = []
    epoch_step_time = []
    

    for step in range(steps_per_epoch):

        step_start_time = time.time()

        # generate inputs (and true outputs) and convert them to tensors

        inputs = [item.to(device) for item in model_input]
        y_true = [item.to(device) for item in model_y_true]
        data_loading_time = time.time()
        print(f"load data to device took{data_loading_time-step_start_time: .2f} seconds")

        # run inputs through the model to produce a warped image and flow field
        y_pred = model(*inputs)
        go_through_model_time = time.time()
        print(f"go through the model took {go_through_model_time-data_loading_time: .2f} seconds")

        # calculate total loss
        loss = 0
        loss_list = []
        for n, loss_function in enumerate(losses):
            curr_loss = loss_function(y_true[n], y_pred[n]) * weights[n]
            loss_list.append(curr_loss.item())
            loss += curr_loss
        epoch_similarity_loss_list.append(loss_list[0])
        epoch_deformation_loss_list.append(loss_list[1])
        loss_calculation_time = time.time()
        print(f"loss calculationt took {loss_calculation_time-go_through_model_time: .2f} seconds")

        epoch_loss.append(loss_list)
        epoch_total_loss.append(loss.item())
        epoch_total_loss_list.append(loss.item())

        # backpropagate and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        backpropagate_time = time.time()
        print(f"backpropagate took {backpropagate_time-loss_calculation_time: .2f} seconds")


        # get compute time
        epoch_step_time.append(time.time() - step_start_time)
    

    # print epoch info
    epoch_info = 'Epoch %d/%d' % (epoch + 1, epochs)
    time_info = 'training %.4f sec/step' % np.mean(epoch_step_time)
    losses_info = ', '.join(['%.4e' % f for f in np.mean(epoch_loss, axis=0)])
    loss_info = 'loss: %.4e  (%s)' % (np.mean(epoch_total_loss), losses_info)
    print(' - '.join((epoch_info, time_info, loss_info)), flush=True)
# final model save
model.save('mice_dataset2/vxm_evaluation_new/385/new_model_optimized_atlas_385.pt')

In [14]:
""" Generating predicted atlas/warpfield
predicted_atlas: predicted_atals_*.nii
predicted_atlas_warpfield: predicted_atlas_warpfield_*.nii
"""

# device handling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#register
#load moving and fixed images
moving = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/385/normalized_atlas_385.nii', add_batch_axis=True, add_feat_axis=True)
fixed, fixed_affine = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/385/normalized_pa_385.nii', add_batch_axis=True, add_feat_axis=True, ret_affine=True)

# load and set up model
model_evaluate = vxm.networks.VxmDense.load('mice_dataset2/vxm_evaluation_new/1e4_batch4/1000.pt', device)
#model_evaluate.tranformer = nnSpatialTransformer((160,128,112))
model_evaluate.to(device)
model_evaluate.eval()

# set up tensors and permute
input_moving = torch.from_numpy(moving).to(device).float().permute(0, 4, 1, 2, 3)
input_fixed = torch.from_numpy(fixed).to(device).float().permute(0, 4, 1, 2, 3)

# predict
moved, warp = model_evaluate(input_moving, input_fixed, registration=True)

# save moved image

moved = moved.detach().cpu().numpy().squeeze()
vxm.py.utils.save_volfile(moved, 'mice_dataset2/vxm_evaluation_new/385/new_model_predicted_atlas_385.nii', fixed_affine)

# save warp
warp = warp.detach().cpu().numpy().squeeze()
vxm.py.utils.save_volfile(warp, 'mice_dataset2/vxm_evaluation_new/385/new_model_predicted_atlas_warpfield_385.nii', fixed_affine)

""" Generating Optimized atlas/warpfield
optimized_atlas: optimized_atlas_*.nii
optimized_warpfield: optimized_atlas_warpfield_*.nii
"""
#register
#load moving and fixed images
moving_2 = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/385/normalized_atlas_385.nii', add_batch_axis=True, add_feat_axis=True)
fixed_2, fixed_affine_2 = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/385/normalized_pa_385.nii', add_batch_axis=True, add_feat_axis=True, ret_affine=True)

# load and set up model
model_evaluate_2 = vxm.networks.VxmDense.load('mice_dataset2/vxm_evaluation_new/385/new_model_optimized_atlas_385.pt', device)

model_evaluate_2.to(device)
model_evaluate_2.eval()

# set up tensors and permute
input_moving_2 = torch.from_numpy(moving_2).to(device).float().permute(0, 4, 1, 2, 3)
input_fixed_2 = torch.from_numpy(fixed_2).to(device).float().permute(0, 4, 1, 2, 3)

# predict
moved_2, warp_2 = model_evaluate_2(input_moving_2, input_fixed_2, registration=True)

# save moved image

moved_2 = moved_2.detach().cpu().numpy().squeeze()
vxm.py.utils.save_volfile(moved_2, 'mice_dataset2/vxm_evaluation_new/385/new_model_optimized_atlas_385.nii', fixed_affine_2)

# save warp
warp_2 = warp_2.detach().cpu().numpy().squeeze()
vxm.py.utils.save_volfile(warp_2, 'mice_dataset2/vxm_evaluation_new/385/new_model_optimized_atlas_warpfield_385.nii', fixed_affine_2)